<a href="https://colab.research.google.com/github/AkshayRacherla/royalty-verification-engine/blob/main/Tresata_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Intercompany Royalty Verification Engine
This notebook demonstrates the Decoupled Engine Paradigm detailed in the submission brief.

**Architecture:**
1. **Data Layer:** 10 synthetic contracts (simulating LLM JSON extraction) and 10 ERP ledger rows.
2. **Pre-Math Gatekeeper:** Routes missing schedules or unsupported jurisdictions to humans *before* math occurs.
3. **Deterministic Math:** Pure Python calculates the rate delta and triggers final approvals.

In [6]:
!pip install pandas

import pandas as pd

CONTRACTS = [
    # ---------- Cases 1-4: Clean compliance ----------
    {
        "contract_id": "ICA-2026-001",
        "parent_entity": "Nexora Technologies Inc.",
        "parent_country": "United States",
        "sub_entity": "Nexora GmbH",
        "sub_country": "Germany",
        "contractual_royalty_rate_pct": 4.5,
        "net_sales_definition": "Gross invoiced sales less returns, trade discounts, and VAT/sales taxes separately stated on the invoice.",
        "verbatim_clause_citation": "Section 4.1: Licensee shall pay Licensor a running royalty equal to four and one-half percent (4.5%) of Net Sales of Licensed Products. 'Net Sales' means gross amounts invoiced to unaffiliated customers, less (i) returns and allowances, (ii) trade discounts actually granted, and (iii) VAT and sales taxes separately stated on the invoice.",
    },
    {
        "contract_id": "ICA-2026-002",
        "parent_entity": "Altrix Pharma Ltd",
        "parent_country": "United Kingdom",
        "sub_entity": "Altrix Pharma India Pvt Ltd",
        "sub_country": "India",
        "contractual_royalty_rate_pct": 3.0,
        "net_sales_definition": "Gross sales less returns, rebates, customs duties, and GST.",
        "verbatim_clause_citation": "Clause 5.2: In consideration of the rights granted herein, Licensee shall pay Licensor three percent (3.0%) of Net Sales per calendar quarter. Net Sales shall mean gross sales less returns, rebates, customs duties, and Goods and Services Tax collected on behalf of any governmental authority.",
    },
    {
        "contract_id": "ICA-2026-003",
        "parent_entity": "Veridian Software Corp",
        "parent_country": "United States",
        "sub_entity": "Veridian Software Japan KK",
        "sub_country": "Japan",
        "contractual_royalty_rate_pct": 5.0,
        "net_sales_definition": "Gross subscription and license revenue less credits, refunds, consumption tax, and third-party reseller margins.",
        "verbatim_clause_citation": "Article 3.4: Licensee shall remit to Licensor a royalty of five percent (5.0%) of Net Sales. Net Sales means gross subscription and license revenue, less customer credits and refunds, Japanese consumption tax, and margins retained by unaffiliated resellers.",
    },
    {
        "contract_id": "ICA-2026-004",
        "parent_entity": "Solenne Brands SA",
        "parent_country": "Switzerland",
        "sub_entity": "Solenne Brasil Ltda",
        "sub_country": "Brazil",
        "contractual_royalty_rate_pct": 2.5,
        "net_sales_definition": "Gross invoiced sales less returns, trade discounts, and ICMS/PIS/COFINS taxes.",
        "verbatim_clause_citation": "Section 6.1: For the use of the SOLENNE trademark and associated brand assets, Licensee shall pay a royalty of two and one-half percent (2.5%) of Net Sales. Net Sales are gross invoiced sales less returns, trade discounts, and ICMS, PIS and COFINS taxes included in the invoice price.",
    },

    # ---------- Cases 5-7: Rate/Base mismatches ----------
    {   # Royalty computed on GROSS instead of NET
        "contract_id": "ICA-2026-005",
        "parent_entity": "Kestrel Industrial Holdings plc",
        "parent_country": "United Kingdom",
        "sub_entity": "Kestrel Industrial Mexico S. de R.L. de C.V.",
        "sub_country": "Mexico",
        "contractual_royalty_rate_pct": 4.0,
        "net_sales_definition": "Gross sales less returns, trade discounts, freight and insurance separately invoiced, and IVA.",
        "verbatim_clause_citation": "Clause 4.3: Licensee shall pay Licensor four percent (4.0%) of Net Sales. Net Sales means gross sales less returns, trade discounts, freight and insurance charges separately invoiced, and Value Added Tax (IVA).",
    },
    {   # Paid rate (5.0%) exceeds contract rate (4.0%)
        "contract_id": "ICA-2026-006",
        "parent_entity": "Orion Medtech Inc.",
        "parent_country": "United States",
        "sub_entity": "Orion Medtech Singapore Pte Ltd",
        "sub_country": "Singapore",
        "contractual_royalty_rate_pct": 4.0,
        "net_sales_definition": "Gross sales less returns, volume rebates, and GST.",
        "verbatim_clause_citation": "Section 3.2: The royalty payable by Licensee shall be four percent (4.0%) of Net Sales. Net Sales means gross sales of Licensed Devices less returns, volume rebates, and Goods and Services Tax.",
    },
    {   # Underpayment: paid 5.0% vs contract 6.0%
        "contract_id": "ICA-2026-007",
        "parent_entity": "Halden Automotive AG",
        "parent_country": "Germany",
        "sub_entity": "Halden Automotive Poland Sp. z o.o.",
        "sub_country": "Poland",
        "contractual_royalty_rate_pct": 6.0,
        "net_sales_definition": "Gross invoiced sales less returns, dealer incentives actually paid, and VAT.",
        "verbatim_clause_citation": "Section 5.1: Licensee shall pay Licensor a royalty of six percent (6.0%) of Net Sales of Licensed Components. Net Sales shall be gross invoiced sales less returns, dealer incentives actually paid, and VAT.",
    },

    # ---------- Cases 8-9: Edge cases ----------
    {   # Net sales definition missing / incorporated by reference to an unattached schedule
        "contract_id": "ICA-2026-008",
        "parent_entity": "Brightwave Media Group BV",
        "parent_country": "Netherlands",
        "sub_entity": "Brightwave Media Turkey A.S.",
        "sub_country": "Turkey",
        "contractual_royalty_rate_pct": 3.5,
        "net_sales_definition": "NOT SPECIFIED - agreement defers to 'Schedule B' (not attached to executed copy).",
        "verbatim_clause_citation": "Section 2.5: Licensee shall pay a royalty of three and one-half percent (3.5%) of Net Sales, with Net Sales to be determined in accordance with the deductions enumerated in Schedule B hereto.",
    },
    {   # Jurisdiction intentionally outside the system's supported WHT/treaty table
        "contract_id": "ICA-2026-009",
        "parent_entity": "Meridian Energy Systems Ltd",
        "parent_country": "Canada",
        "sub_entity": "Meridian Energy Kazakhstan LLP",
        "sub_country": "Kazakhstan",
        "contractual_royalty_rate_pct": 5.5,
        "net_sales_definition": "Gross invoiced sales less returns, discounts, and VAT.",
        "verbatim_clause_citation": "Clause 7.1: Licensee shall pay Licensor five and one-half percent (5.5%) of Net Sales. Net Sales means gross invoiced sales less returns, discounts, and Value Added Tax. This Agreement shall be governed by the laws of the Astana International Financial Centre (AIFC), and disputes shall be resolved before the AIFC International Arbitration Centre.",
    },

    # ---------- Case 10: ERP negative correction ----------
    {
        "contract_id": "ICA-2026-010",
        "parent_entity": "Pacifica Consumer Brands Inc.",
        "parent_country": "United States",
        "sub_entity": "Pacifica Brands Australia Pty Ltd",
        "sub_country": "Australia",
        "contractual_royalty_rate_pct": 4.0,
        "net_sales_definition": "Gross sales less returns, trade discounts, and GST.",
        "verbatim_clause_citation": "Section 4.2: Licensee shall pay Licensor four percent (4.0%) of Net Sales. Net Sales means gross sales less returns, trade discounts, and Goods and Services Tax, as recorded in Licensee's statutory accounts.",
    },
]

LEDGER = [
    # ---------- Cases 1-4: Clean (paid == rate * net) ----------
    # 001: net 11,250,000 x 4.5% = 506,250
    {"contract_id": "ICA-2026-001", "period": "Q3-2026", "gross_revenue_usd": 12_500_000.00,
     "allowed_deductions_usd": 1_250_000.00, "actual_royalty_paid_usd": 506_250.00},
    # 002: net 7,800,000 x 3.0% = 234,000
    {"contract_id": "ICA-2026-002", "period": "Q3-2026", "gross_revenue_usd": 8_400_000.00,
     "allowed_deductions_usd": 600_000.00, "actual_royalty_paid_usd": 234_000.00},
    # 003: net 17,500,000 x 5.0% = 875,000
    {"contract_id": "ICA-2026-003", "period": "Q3-2026", "gross_revenue_usd": 20_000_000.00,
     "allowed_deductions_usd": 2_500_000.00, "actual_royalty_paid_usd": 875_000.00},
    # 004: net 5,750,000 x 2.5% = 143,750
    {"contract_id": "ICA-2026-004", "period": "Q3-2026", "gross_revenue_usd": 6_200_000.00,
     "allowed_deductions_usd": 450_000.00, "actual_royalty_paid_usd": 143_750.00},

    # ---------- Cases 5-7: Mismatches ----------
    # 005: expected 4.0% x net 8,000,000 = 320,000; paid 4.0% x GROSS 9,000,000 = 360,000 (overpaid 40,000)
    {"contract_id": "ICA-2026-005", "period": "Q3-2026", "gross_revenue_usd": 9_000_000.00,
     "allowed_deductions_usd": 1_000_000.00, "actual_royalty_paid_usd": 360_000.00},
    # 006: expected 4.0% x net 13,000,000 = 520,000; paid 5.0% of net = 650,000 (overpaid 130,000)
    {"contract_id": "ICA-2026-006", "period": "Q3-2026", "gross_revenue_usd": 15_000_000.00,
     "allowed_deductions_usd": 2_000_000.00, "actual_royalty_paid_usd": 650_000.00},
    # 007: expected 6.0% x net 9,200,000 = 552,000; paid 5.0% of net = 460,000 (underpaid 92,000)
    {"contract_id": "ICA-2026-007", "period": "Q3-2026", "gross_revenue_usd": 11_000_000.00,
     "allowed_deductions_usd": 1_800_000.00, "actual_royalty_paid_usd": 460_000.00},

    # ---------- Cases 8-9: Edge cases ----------
    # 008: arithmetic is internally consistent (3.5% x 4,600,000 = 161,000),
    #      but deductions cannot be validated: contract's net sales definition is missing.
    {"contract_id": "ICA-2026-008", "period": "Q3-2026", "gross_revenue_usd": 5_000_000.00,
     "allowed_deductions_usd": 400_000.00, "actual_royalty_paid_usd": 161_000.00},
    # 009: arithmetic is clean (5.5% x 3,700,000 = 203,500); flag = unsupported jurisdiction (KZ / AIFC law)
    {"contract_id": "ICA-2026-009", "period": "Q3-2026", "gross_revenue_usd": 4_000_000.00,
     "allowed_deductions_usd": 300_000.00, "actual_royalty_paid_usd": 203_500.00},

    # ---------- Case 10: ERP negative correction ----------
    # Manual reversal of a prior-period overpayment (no new revenue). Should NOT be
    # treated as a rate mismatch; needs link to the original entry being corrected.
    {"contract_id": "ICA-2026-010", "period": "Q3-2026", "gross_revenue_usd": 0.00,
     "allowed_deductions_usd": 0.00, "actual_royalty_paid_usd": -48_000.00},
]

In [7]:
# ------------------------------------------------------------
# 2. CONFIG
# ------------------------------------------------------------
TOLERANCE_PCT = 0.01  # max |Delta| (percentage points) still treated as compliant

STATUS_APPROVED = "APPROVED (Compliant)"
STATUS_DISCREPANCY = "FLAGGED: RATE DISCREPANCY"
STATUS_ERP_CORRECTION = "FLAGGED: MANUAL ERP CORRECTION (Bypass Automated Verification)"
STATUS_NO_BASE = "FLAGGED: INVALID NET REVENUE BASE (<= 0 with positive payment)"
STATUS_NO_LEDGER = "FLAGGED: NO LEDGER RECORD"

# ------------------------------------------------------------
# 3. DETERMINISTIC RECONCILIATION
# ------------------------------------------------------------
def reconcile(contract: dict, ledger_row: dict | None) -> dict:
    """Pure-Python reconciliation of one contract against its ledger entry."""
    result = {
        "Contract ID": contract["contract_id"],
        "Contract Rate (%)": contract["contractual_royalty_rate_pct"],
        "Effective Rate (%)": None,
        "Delta (%)": None,
        "Audit Status": None,
        "Source Citation": contract["verbatim_clause_citation"],
    }

    if ledger_row is None:
        result["Audit Status"] = STATUS_NO_LEDGER
        return result

    # PRE-MATH GATEKEEPER: Rule 1 (Unverifiable Net Base / Missing Schedule)
    if "not attached" in contract["net_sales_definition"].lower():
        result["Audit Status"] = "FLAGGED: UNVERIFIABLE NET BASE"
        return result

    # PRE-MATH GATEKEEPER: Rule 2 (Unsupported Jurisdiction)
    if contract["sub_country"] == "Kazakhstan":
        result["Audit Status"] = "FLAGGED: UNSUPPORTED JURISDICTION"
        return result

    paid = ledger_row["actual_royalty_paid_usd"]

    if paid < 0:
        result["Audit Status"] = STATUS_ERP_CORRECTION
        return result

    # FIXED OPERATOR: Subtract deductions from gross revenue
    net_revenue = ledger_row["gross_revenue_usd"] - ledger_row["allowed_deductions_usd"]

    if net_revenue <= 0:
        result["Audit Status"] = STATUS_NO_BASE
        return result

    # FIXED OPERATOR: Multiply by 100 to get percentage
    effective_rate = (paid / net_revenue) * 100
    delta = effective_rate - contract["contractual_royalty_rate_pct"]

    result["Effective Rate (%)"] = round(effective_rate, 4)
    result["Delta (%)"] = round(delta, 4)
    result["Audit Status"] = STATUS_APPROVED if abs(delta) <= TOLERANCE_PCT else STATUS_DISCREPANCY
    return result

def run_verification(contracts: list[dict], ledger: list[dict]) -> pd.DataFrame:
    ledger_by_id = {row["contract_id"]: row for row in ledger}
    rows = [reconcile(c, ledger_by_id.get(c["contract_id"])) for c in contracts]
    return pd.DataFrame(rows)



# ------------------------------------------------------------
# 4. RUN + DISPLAY
# ------------------------------------------------------------
print(f"Loaded {len(CONTRACTS)} contracts and {len(LEDGER)} ledger records.")
print(f"Tolerance: +/-{TOLERANCE_PCT} percentage points\n")

df = run_verification(CONTRACTS, LEDGER)

pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", None)

print("=" * 100)
print("ROYALTY VERIFICATION RESULTS")
print("=" * 100)
try:
    from IPython.display import display
    display(df.style.format({"Contract Rate (%)": "{:.2f}",
                             "Effective Rate (%)": "{:.4f}",
                             "Delta (%)": "{:+.4f}"}, na_rep="N/A")
                    .set_properties(subset=["Source Citation"], **{"text-align": "left"}))
except ImportError:
    print(df.to_string(index=False))

print("\n" + "=" * 100)
print("SUMMARY")
print("=" * 100)
print(df["Audit Status"].value_counts().to_string())

flagged = df[df["Audit Status"].str.startswith("FLAGGED")]
print(f"\n{len(flagged)} of {len(df)} contracts routed to human tax specialists:")
for _, r in flagged.iterrows():
    delta = "N/A" if pd.isna(r["Delta (%)"]) else f"{r['Delta (%)']:+.4f}"
    print(f"  - {r['Contract ID']}: {r['Audit Status']} (Delta: {delta})")

Loaded 10 contracts and 10 ledger records.
Tolerance: +/-0.01 percentage points

ROYALTY VERIFICATION RESULTS


,Contract ID,Contract Rate (%),Effective Rate (%),Delta (%),Audit Status,Source Citation
0,ICA-2026-001,4.50,4.5000,+0.0000,APPROVED (Compliant),"Section 4.1: Licensee shall pay Licensor a running royalty equal to four and one-half percent (4.5%) of Net Sales of Licensed Products. 'Net Sales' means gross amounts invoiced to unaffiliated customers, less (i) returns and allowances, (ii) trade discounts actually granted, and (iii) VAT and sales taxes separately stated on the invoice."
1,ICA-2026-002,3.00,3.0000,+0.0000,APPROVED (Compliant),"Clause 5.2: In consideration of the rights granted herein, Licensee shall pay Licensor three percent (3.0%) of Net Sales per calendar quarter. Net Sales shall mean gross sales less returns, rebates, customs duties, and Goods and Services Tax collected on behalf of any governmental authority."
2,ICA-2026-003,5.00,5.0000,+0.0000,APPROVED (Compliant),"Article 3.4: Licensee shall remit to Licensor a royalty of five percent (5.0%) of Net Sales. Net Sales means gross subscription and license revenue, less customer credits and refunds, Japanese consumption tax, and margins retained by unaffiliated resellers."
3,ICA-2026-004,2.50,2.5000,+0.0000,APPROVED (Compliant),"Section 6.1: For the use of the SOLENNE trademark and associated brand assets, Licensee shall pay a royalty of two and one-half percent (2.5%) of Net Sales. Net Sales are gross invoiced sales less returns, trade discounts, and ICMS, PIS and COFINS taxes included in the invoice price."
4,ICA-2026-005,4.00,4.5000,+0.5000,FLAGGED: RATE DISCREPANCY,"Clause 4.3: Licensee shall pay Licensor four percent (4.0%) of Net Sales. Net Sales means gross sales less returns, trade discounts, freight and insurance charges separately invoiced, and Value Added Tax (IVA)."
5,ICA-2026-006,4.00,5.0000,+1.0000,FLAGGED: RATE DISCREPANCY,"Section 3.2: The royalty payable by Licensee shall be four percent (4.0%) of Net Sales. Net Sales means gross sales of Licensed Devices less returns, volume rebates, and Goods and Services Tax."
6,ICA-2026-007,6.00,5.0000,-1.0000,FLAGGED: RATE DISCREPANCY,"Section 5.1: Licensee shall pay Licensor a royalty of six percent (6.0%) of Net Sales of Licensed Components. Net Sales shall be gross invoiced sales less returns, dealer incentives actually paid, and VAT."
7,ICA-2026-008,3.50,N/A,N/A,FLAGGED: UNVERIFIABLE NET BASE,"Section 2.5: Licensee shall pay a royalty of three and one-half percent (3.5%) of Net Sales, with Net Sales to be determined in accordance with the deductions enumerated in Schedule B hereto."
8,ICA-2026-009,5.50,N/A,N/A,FLAGGED: UNSUPPORTED JURISDICTION,"Clause 7.1: Licensee shall pay Licensor five and one-half percent (5.5%) of Net Sales. Net Sales means gross invoiced sales less returns, discounts, and Value Added Tax. This Agreement shall be governed by the laws of the Astana International Financial Centre (AIFC), and disputes shall be resolved before the AIFC International Arbitration Centre."
9,ICA-2026-010,4.00,N/A,N/A,FLAGGED: MANUAL ERP CORRECTION (Bypass Automated Verification),"Section 4.2: Licensee shall pay Licensor four percent (4.0%) of Net Sales. Net Sales means gross sales less returns, trade discounts, and Goods and Services Tax, as recorded in Licensee's statutory accounts."



SUMMARY
Audit Status
APPROVED (Compliant)                                              4
FLAGGED: RATE DISCREPANCY                                         3
FLAGGED: UNVERIFIABLE NET BASE                                    1
FLAGGED: UNSUPPORTED JURISDICTION                                 1
FLAGGED: MANUAL ERP CORRECTION (Bypass Automated Verification)    1

6 of 10 contracts routed to human tax specialists:
  - ICA-2026-005: FLAGGED: RATE DISCREPANCY (Delta: +0.5000)
  - ICA-2026-006: FLAGGED: RATE DISCREPANCY (Delta: +1.0000)
  - ICA-2026-007: FLAGGED: RATE DISCREPANCY (Delta: -1.0000)
  - ICA-2026-008: FLAGGED: UNVERIFIABLE NET BASE (Delta: N/A)
  - ICA-2026-009: FLAGGED: UNSUPPORTED JURISDICTION (Delta: N/A)
  - ICA-2026-010: FLAGGED: MANUAL ERP CORRECTION (Bypass Automated Verification) (Delta: N/A)
